In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder
from imblearn.under_sampling import NearMiss
from imblearn.over_sampling import SMOTE
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
import warnings
warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\fix_clean_data.csv")
df.head()

,id,text,durasi,emotion
0,1,tradisional laos kayak gimana kayak gini jadi ...,1.58,6
1,2,episode berikutnya akan dimulai sekarang kita ...,2.07,6
2,3,itu agak aneh ya biasanya pohon pepaya itu kan...,0.45,6
3,4,bulan ini sampah organik di rumah kita olah pa...,2.06,6
4,5,mobil yang akan kita pakai buat root party nan...,1.04,4


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 722 entries, 0 to 721
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   id       722 non-null    int64  
 1   text     722 non-null    object 
 2   durasi   722 non-null    float64
 3   emotion  722 non-null    int64  
dtypes: float64(1), int64(2), object(1)
memory usage: 22.7+ KB


In [19]:
# Download NLTK data (run once)
try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)
except:
    pass

In [33]:
# 2. Ambil teks
texts = df["text"].astype(str)

# 3. Buat TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,       # batas jumlah fitur
    ngram_range=(1,2),       # unigram + bigram (opsional)
    stop_words=None  # hapus stopwords bahasa Indonesia
)

# 4. Transformasi
X_tfidf = tfidf.fit_transform(texts)

# 5. Masukkan hasil vektor ke dataframe (simpan sebagai list per baris)
df["vector"] = list(X_tfidf.toarray())

# 6. Simpan ke file baru jika perlu
df.to_csv("data_tfidf.csv", index=False)

print("Jumlah fitur:", len(tfidf.get_feature_names_out()))
print("Contoh fitur:", tfidf.get_feature_names_out()[:50])


print(df.head())

Jumlah fitur: 5000
Contoh fitur: ['abis' 'abis itu' 'about' 'abuabu' 'ac' 'acara' 'acaranya' 'aceh'
 'aceh sumut' 'acnya' 'action' 'active' 'activity' 'ada' 'ada ada'
 'ada aja' 'ada apa' 'ada banget' 'ada banyak' 'ada beberapa' 'ada bisa'
 'ada buat' 'ada dan' 'ada di' 'ada dua' 'ada efek' 'ada enggak'
 'ada fitur' 'ada juga' 'ada lagi' 'ada masalah' 'ada mazda' 'ada mobil'
 'ada orang' 'ada promo' 'ada rasa' 'ada satu' 'ada sedikit' 'ada semua'
 'ada shade' 'ada tapi' 'ada vitamin' 'ada warna' 'ada yang' 'adalah'
 'adalah di' 'adalah kalau' 'adalah mobil' 'adalah salah' 'adalah warna']
   id                                               text  durasi  emotion  \
0   1  tradisional laos kayak gimana kayak gini jadi ...    1.58        6   
1   2  episode berikutnya akan dimulai sekarang kita ...    2.07        6   
2   3  itu agak aneh ya biasanya pohon pepaya itu kan...    0.45        6   
3   4  bulan ini sampah organik di rumah kita olah pa...    2.06        6   
4   5  mobil yang ak

In [37]:
import numpy as np

row0 = X_tfidf[0].toarray().flatten()
nonzero_idx = np.where(row0 > 0)[0]

print("=== TF-IDF untuk baris 0 ===")
for i in nonzero_idx:
    print(tfidf.get_feature_names_out()[i], ":", row0[i])


=== TF-IDF untuk baris 0 ===
ada : 0.10077824074203662
ada yang : 0.03773485177280357
adalah : 0.07266003548409587
apa : 0.026528810383794718
apa ya : 0.059078680914385104
atau : 0.042668888968698616
bagian : 0.03471075570706504
bagus : 0.0335815648925438
banget : 0.041676780710054456
banget buat : 0.043572502984199034
bawa : 0.043572502984199034
bawah : 0.06344348749750679
bawahnya : 0.060370591021358486
bedanya : 0.11184644659510659
begitulah : 0.06352604863819028
berapa : 0.03963068826216343
berbagai : 0.05147585557374811
besar : 0.03673702250937559
biar : 0.02978923108310317
biar lebih : 0.05792302794368586
bikin : 0.029375274090996318
bisa : 0.01543870612361634
buah : 0.050882816098871485
buat : 0.08596680998334336
buat kalian : 0.04387303023311112
buat temanteman : 0.05347566021988068
buat yang : 0.054232412561050936
bukan : 0.03004531800895636
cari : 0.03942566250930593
coba : 0.03085153405702221
coba komen : 0.04587283487924369
cocok : 0.0384543593976592
cocok banget : 0.047872